# Credit Card Fraud Detection — Autoencoder-based Anomaly Detection

**University Deep Learning Project**

This notebook builds a deep **Autoencoder** that learns the manifold of *normal* credit-card
transactions. Fraudulent transactions are then detected as **reconstruction-error outliers**:
the autoencoder, having only ever seen normal data, fails to reconstruct frauds well, so their
MSE is large.

### Why an Autoencoder (and not a classifier)?

The Kaggle Credit Card dataset is **extremely imbalanced** (~0.17% fraud). A vanilla classifier
either ignores the minority class or overfits to it. By training the AE on *normal* data only,
we sidestep the imbalance entirely and turn fraud detection into **unsupervised anomaly
detection** on top of a learned representation.

### Pipeline

1. Load the already-preprocessed (StandardScaler-normalized) dataset.
2. Separate normal (Class=0) and fraud (Class=1) transactions.
3. Carve out three disjoint sets:
   - **Train** — normal-only data the AE will learn from.
   - **Validation** — normal + fraud, used for picking the anomaly threshold.
   - **Test** — normal + fraud, used **once** for final reporting.
4. Build a regularized AE (BatchNorm + light Dropout + L2) with proper callbacks.
5. Compute per-sample reconstruction MSE and pick a threshold using a **recall-aware**
   strategy (F2-optimal by default, with a recall-floor fallback) — appropriate for fraud,
   where false negatives cost far more than false positives.
6. Evaluate on the held-out test set: Precision, Recall, F1, Confusion Matrix, ROC-AUC, PR-AUC.
7. Save the trained model, threshold, and a Flask-friendly inference helper.

## 1. Imports, reproducibility, and global configuration

In [ ]:
import os
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import Model, Input, regularizers
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
)

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

# Reproducibility — fixing seeds across libraries makes runs comparable.
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Project paths. The CSV is expected to sit next to this notebook.
PROJECT_DIR = Path.cwd()
DATA_PATH = PROJECT_DIR / "processed_creditcard.csv"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)

MODEL_H5_PATH       = ARTIFACTS_DIR / "autoencoder_fraud.h5"
MODEL_KERAS_PATH    = ARTIFACTS_DIR / "autoencoder_fraud.keras"
THRESHOLD_JSON_PATH = ARTIFACTS_DIR / "threshold.json"
HISTORY_JSON_PATH   = ARTIFACTS_DIR / "history.json"

print("TensorFlow version :", tf.__version__)
print("GPU available      :", bool(tf.config.list_physical_devices("GPU")))
print("Data path          :", DATA_PATH)
print("Artifacts dir      :", ARTIFACTS_DIR)

## 2. Load the preprocessed dataset

The preprocessing notebook already standardised every numeric column (Time, V1–V28, Amount)
with `StandardScaler`. We just load it, sanity-check the schema, and inspect the class balance.

In [ ]:
assert DATA_PATH.exists(), f"Could not find {DATA_PATH}. Place processed_creditcard.csv next to this notebook."

df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
# Quick integrity checks.
assert "Class" in df.columns, "Target column 'Class' missing."
assert df.isnull().sum().sum() == 0, "Dataset contains NaNs — preprocessing step seems off."

class_counts = df["Class"].value_counts().sort_index()
fraud_ratio  = class_counts.get(1, 0) / len(df)

print("Class distribution:")
print(class_counts.to_string())
print(f"\nFraud ratio: {fraud_ratio:.4%}  (≈ 1 fraud per {1/fraud_ratio:,.0f} txns)")

## 3. Split strategy

Because the AE must be trained on *normal* transactions only, we split the two classes
separately and recombine them where appropriate:

| Split | Composition | Used for |
|-------|-------------|----------|
| `X_train` | 70% of normals | Training the autoencoder |
| `X_val_normal` | 15% of normals | Early-stopping (monitor reconstruction MSE) |
| `X_val_full` = val normals + 50% of frauds | mixed | **Choosing the anomaly threshold** |
| `X_test_full` = test normals + 50% of frauds | mixed | **Final reporting only** |

No fraud is ever seen during AE training. The two fraud halves are disjoint, so the
threshold-selection set and the test set are independent.

In [ ]:
FEATURES = [c for c in df.columns if c != "Class"]
N_FEATURES = len(FEATURES)
print(f"Input features ({N_FEATURES}): {FEATURES}")

normal_df = df[df["Class"] == 0].reset_index(drop=True)
fraud_df  = df[df["Class"] == 1].reset_index(drop=True)
print(f"Normals: {len(normal_df):,}   Frauds: {len(fraud_df):,}")

In [ ]:
# Step A — split normals: 70% train, 15% val, 15% test.
normal_train, normal_temp = train_test_split(
    normal_df, test_size=0.30, random_state=SEED, shuffle=True
)
normal_val, normal_test = train_test_split(
    normal_temp, test_size=0.50, random_state=SEED, shuffle=True
)

# Step B — split frauds 50/50 between threshold-selection set and final test set.
fraud_val, fraud_test = train_test_split(
    fraud_df, test_size=0.50, random_state=SEED, shuffle=True, stratify=None
)

# Step C — assemble matrices.
X_train      = normal_train[FEATURES].values.astype(np.float32)
y_train      = np.zeros(len(X_train), dtype=np.int32)

X_val_full   = pd.concat([normal_val, fraud_val], ignore_index=True)
X_val_full   = X_val_full.sample(frac=1, random_state=SEED).reset_index(drop=True)
y_val_full   = X_val_full["Class"].values.astype(np.int32)
X_val_full   = X_val_full[FEATURES].values.astype(np.float32)

X_test_full  = pd.concat([normal_test, fraud_test], ignore_index=True)
X_test_full  = X_test_full.sample(frac=1, random_state=SEED).reset_index(drop=True)
y_test_full  = X_test_full["Class"].values.astype(np.int32)
X_test_full  = X_test_full[FEATURES].values.astype(np.float32)

# Validation-of-normals only — used during training for early stopping.
X_val_normal = normal_val[FEATURES].values.astype(np.float32)

print(f"X_train       : {X_train.shape}   (all normals)")
print(f"X_val_normal  : {X_val_normal.shape}   (normals only, monitored during training)")
print(f"X_val_full    : {X_val_full.shape}    frauds={y_val_full.sum()}   (threshold tuning)")
print(f"X_test_full   : {X_test_full.shape}    frauds={y_test_full.sum()}  (final evaluation)")

## 4. Autoencoder architecture

A compact, regularized symmetric AE:

```
Input(30) → Dense(28, relu, L2) → BN → Dropout(0.10)
          → Dense(20, relu, L2) → BN → Dropout(0.10)
          → Dense(12, relu, L2) → BN
          → Dense(7,  relu)              ← bottleneck
          → Dense(12, relu, L2) → BN
          → Dense(20, relu, L2) → BN → Dropout(0.10)
          → Dense(28, relu, L2) → BN → Dropout(0.10)
          → Dense(30, linear)            ← reconstruction
```

Design choices (tuned for higher recall):
- **Lower dropout (0.10)** lets the AE fit the normal manifold more tightly, so frauds
  reconstruct visibly worse — the gap between the two error distributions widens.
- **Three encoder stages with a tighter bottleneck (7 units)** force a more compact
  normal representation, pushing out-of-distribution frauds even further from the manifold.
- **BatchNorm + L2 weight decay** keep training stable and prevent overfitting.
- **Linear output** because targets are standardised (zero-mean, real-valued).
- **Adam + MSE** is the standard reliable choice for reconstruction.
- Architecture is **lightweight** (~few thousand params) and trains in seconds on GPU.

In [ ]:
def build_autoencoder(
    n_features: int,
    encoder_units=(28, 20, 12),
    bottleneck_units: int = 7,
    dropout_rate: float = 0.10,
    l2_lambda: float = 1e-5,
    learning_rate: float = 1e-3,
) -> Model:
    """Build a symmetric deep autoencoder for tabular anomaly detection.

    The decoder mirrors the encoder. BatchNorm stabilises training; light Dropout +
    L2 regularise without preventing a tight fit of the normal manifold (a tight fit
    is exactly what makes frauds stand out in reconstruction error).
    """
    reg = regularizers.l2(l2_lambda)

    inputs = Input(shape=(n_features,), name="input")
    x = inputs

    # Encoder. We drop Dropout in the deepest encoder stage to allow a precise
    # representation right before the bottleneck.
    for i, units in enumerate(encoder_units):
        x = Dense(units, activation="relu", kernel_regularizer=reg, name=f"enc_dense_{i+1}")(x)
        x = BatchNormalization(name=f"enc_bn_{i+1}")(x)
        if i < len(encoder_units) - 1:
            x = Dropout(dropout_rate, name=f"enc_drop_{i+1}")(x)

    # Bottleneck — compact representation of the normal manifold.
    bottleneck = Dense(bottleneck_units, activation="relu", name="bottleneck")(x)

    # Decoder (mirror of encoder); symmetrically, the first decoder layer has no Dropout.
    x = bottleneck
    decoder_units = list(reversed(encoder_units))
    for i, units in enumerate(decoder_units):
        x = Dense(units, activation="relu", kernel_regularizer=reg, name=f"dec_dense_{i+1}")(x)
        x = BatchNormalization(name=f"dec_bn_{i+1}")(x)
        if i > 0:
            x = Dropout(dropout_rate, name=f"dec_drop_{i+1}")(x)

    outputs = Dense(n_features, activation="linear", name="reconstruction")(x)

    model = Model(inputs, outputs, name="fraud_autoencoder")
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss="mse", metrics=["mae"])
    return model


# Build fresh model and discard any previously trained weights.
tf.keras.backend.clear_session()
tf.random.set_seed(SEED)
autoencoder = build_autoencoder(N_FEATURES)
autoencoder.summary()

## 5. Training with callbacks

We monitor `val_loss` on the **normal-only validation set** (`X_val_normal`). Three callbacks:

- `EarlyStopping` — restores the best weights and prevents wasted epochs.
- `ReduceLROnPlateau` — halves the learning rate when validation loss plateaus.
- `ModelCheckpoint` — persists the best epoch to disk during training.

In [ ]:
EPOCHS = 150
BATCH_SIZE = 512

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=15,
        min_delta=1e-5,
        restore_best_weights=True,
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1,
    ),
    ModelCheckpoint(
        filepath=str(MODEL_KERAS_PATH),
        monitor="val_loss",
        save_best_only=True,
        verbose=0,
    ),
]

history = autoencoder.fit(
    X_train, X_train,
    validation_data=(X_val_normal, X_val_normal),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True,
    callbacks=callbacks,
    verbose=2,
)

## 6. Training curves

If the train and validation curves track each other and both decrease, the AE is generalising.
A growing gap would indicate overfitting — our regularisation pushes that back.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history.history["loss"],     label="train MSE")
axes[0].plot(history.history["val_loss"], label="val MSE")
axes[0].set_title("Reconstruction loss (MSE)")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("MSE"); axes[0].legend()

axes[1].plot(history.history["mae"],     label="train MAE")
axes[1].plot(history.history["val_mae"], label="val MAE")
axes[1].set_title("Reconstruction loss (MAE)")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("MAE"); axes[1].legend()

plt.tight_layout(); plt.show()

# Persist training history alongside the model for later analysis.
with open(HISTORY_JSON_PATH, "w") as f:
    json.dump({k: [float(v) for v in vals] for k, vals in history.history.items()}, f, indent=2)

## 7. Reconstruction error & threshold selection

For every sample, the **reconstruction error** is the mean-squared error between the input
and its reconstruction. Normals (which the AE was trained on) reconstruct well — small error.
Frauds reconstruct poorly — large error.

### Which threshold to pick?

In fraud detection a missed fraud (false negative) is **much** more expensive than a false
alarm. We therefore compute **three** threshold candidates on the validation set and let the
business policy choose:

| Strategy | What it optimises | When to use |
|---|---|---|
| **F1-optimal** | Equal weight to precision and recall | Default benchmark |
| **F2-optimal** | Weights recall 4× more than precision | **Default for fraud** ⭐ |
| **Recall ≥ 0.90** | Lowest threshold giving ≥ 90% recall on validation | Strict policy |

We save the **F2-optimal** threshold as the production threshold but evaluate all three on
the test set for transparency.

In [ ]:
def reconstruction_error(model: Model, X: np.ndarray, batch_size: int = 4096) -> np.ndarray:
    """Per-sample MSE between input and its reconstruction."""
    X_hat = model.predict(X, batch_size=batch_size, verbose=0)
    return np.mean(np.square(X - X_hat), axis=1)


# Errors on the validation set (used to choose the threshold).
val_errors  = reconstruction_error(autoencoder, X_val_full)

# Errors on the test set (only used for final reporting below).
test_errors = reconstruction_error(autoencoder, X_test_full)

# Sanity: normal samples should on average have lower error than frauds.
print("Mean error on val NORMAL :", val_errors[y_val_full == 0].mean())
print("Mean error on val FRAUD  :", val_errors[y_val_full == 1].mean())

In [ ]:
def fbeta(p: np.ndarray, r: np.ndarray, beta: float) -> np.ndarray:
    """F_beta score. beta>1 weights recall more, beta<1 weights precision more."""
    b2 = beta * beta
    return (1 + b2) * p * r / (b2 * p + r + 1e-12)


def candidate_thresholds(y_true: np.ndarray, errors: np.ndarray) -> dict:
    """Return several threshold candidates plus their validation metrics."""
    precisions, recalls, thresholds = precision_recall_curve(y_true, errors)
    # precision_recall_curve returns len(thresholds) == len(precisions) - 1.
    p, r, t = precisions[:-1], recalls[:-1], thresholds

    f1 = fbeta(p, r, 1.0)
    f2 = fbeta(p, r, 2.0)

    out = {}

    # F1-optimal
    i = int(np.nanargmax(f1))
    out["f1_optimal"] = {"threshold": float(t[i]), "precision": float(p[i]),
                         "recall": float(r[i]), "f1": float(f1[i]), "f2": float(f2[i])}

    # F2-optimal (recall-weighted) -- our production choice for fraud
    i = int(np.nanargmax(f2))
    out["f2_optimal"] = {"threshold": float(t[i]), "precision": float(p[i]),
                         "recall": float(r[i]), "f1": float(f1[i]), "f2": float(f2[i])}

    # Recall floor at 0.90 -- the *largest* threshold whose recall is still >= 0.90.
    # Larger threshold => higher precision, so we walk from the right.
    mask = r >= 0.90
    if mask.any():
        i = int(np.where(mask)[0].max())
        out["recall_at_least_0.90"] = {
            "threshold": float(t[i]), "precision": float(p[i]),
            "recall": float(r[i]), "f1": float(f1[i]), "f2": float(f2[i]),
        }
    else:
        out["recall_at_least_0.90"] = None

    out["_pr_curve"] = (precisions, recalls, thresholds)
    return out


val_candidates = candidate_thresholds(y_val_full, val_errors)

print(f"{'Strategy':<25s} {'threshold':>10s} {'precision':>10s} {'recall':>10s} {'F1':>8s} {'F2':>8s}")
print("-" * 75)
for name in ["f1_optimal", "f2_optimal", "recall_at_least_0.90"]:
    c = val_candidates[name]
    if c is None:
        print(f"{name:<25s} {'(not reachable on this validation set)':>50s}")
    else:
        print(f"{name:<25s} {c['threshold']:>10.4f} {c['precision']:>10.4f} "
              f"{c['recall']:>10.4f} {c['f1']:>8.4f} {c['f2']:>8.4f}")

# Production threshold = F2-optimal (recall-weighted) -- right choice for fraud.
val_best  = val_candidates["f2_optimal"]
THRESHOLD = val_best["threshold"]
print(f"\n>> Production threshold (F2-optimal on validation): {THRESHOLD:.6f}")

## 8. Final evaluation on the held-out test set

The thresholds are now **frozen** — they were picked using the validation set only. We apply
each candidate to the test set (which the AE has never seen and which contributed nothing to
threshold selection) and report all standard metrics. The **F2-optimal** threshold is the
production default; the others are shown for comparison.

In [ ]:
def evaluate_at(threshold: float, label: str) -> dict:
    """Evaluate the model on the test set at a given threshold."""
    y_pred = (test_errors > threshold).astype(np.int32)
    cm_ = confusion_matrix(y_test_full, y_pred)
    tn_, fp_, fn_, tp_ = cm_.ravel()
    return {
        "label":     label,
        "threshold": float(threshold),
        "precision": float(precision_score(y_test_full, y_pred, zero_division=0)),
        "recall":    float(recall_score(y_test_full, y_pred, zero_division=0)),
        "f1":        float(f1_score(y_test_full, y_pred, zero_division=0)),
        "accuracy":  float((tn_ + tp_) / (tn_ + fp_ + fn_ + tp_)),
        "cm":        {"TN": int(tn_), "FP": int(fp_), "FN": int(fn_), "TP": int(tp_)},
        "y_pred":    y_pred,
    }


# Ranking-based metrics depend only on the error scores, not on the threshold.
roc_auc = roc_auc_score(y_test_full, test_errors)
pr_auc  = average_precision_score(y_test_full, test_errors)

results = {
    "f1_optimal":           evaluate_at(val_candidates["f1_optimal"]["threshold"],          "F1-optimal"),
    "f2_optimal":           evaluate_at(val_candidates["f2_optimal"]["threshold"],          "F2-optimal (production)"),
}
if val_candidates["recall_at_least_0.90"] is not None:
    results["recall_at_least_0.90"] = evaluate_at(
        val_candidates["recall_at_least_0.90"]["threshold"], "Recall >= 0.90"
    )

print("=" * 78)
print("                FINAL TEST-SET PERFORMANCE")
print("=" * 78)
print(f"Ranking metrics (threshold-independent):  ROC-AUC = {roc_auc:.4f}   PR-AUC = {pr_auc:.4f}\n")

header = f"{'Strategy':<26s} {'thr':>8s} {'acc':>7s} {'prec':>7s} {'recall':>8s} {'F1':>7s}   {'TN':>6s} {'FP':>5s} {'FN':>4s} {'TP':>4s}"
print(header); print("-" * len(header))
for r in results.values():
    print(f"{r['label']:<26s} {r['threshold']:>8.3f} {r['accuracy']:>7.4f} "
          f"{r['precision']:>7.4f} {r['recall']:>8.4f} {r['f1']:>7.4f}   "
          f"{r['cm']['TN']:>6d} {r['cm']['FP']:>5d} {r['cm']['FN']:>4d} {r['cm']['TP']:>4d}")

# Promote the production (F2-optimal) result to top-level names for downstream cells.
prod        = results["f2_optimal"]
y_pred_test = prod["y_pred"]
precision   = prod["precision"]
recall      = prod["recall"]
f1          = prod["f1"]
accuracy    = prod["accuracy"]
cm          = np.array([[prod["cm"]["TN"], prod["cm"]["FP"]],
                        [prod["cm"]["FN"], prod["cm"]["TP"]]])
tn, fp, fn, tp = prod["cm"]["TN"], prod["cm"]["FP"], prod["cm"]["FN"], prod["cm"]["TP"]
THRESHOLD = prod["threshold"]

print("\nClassification report at production (F2-optimal) threshold:")
print(classification_report(y_test_full, y_pred_test, target_names=["Normal", "Fraud"], digits=4))

## 9. Visualisations

In [ ]:
# 9a. Reconstruction-error distribution (log-y for visibility) with the chosen threshold.
plt.figure(figsize=(11, 5))
plt.hist(test_errors[y_test_full == 0], bins=80, alpha=0.6, label="Normal", density=True)
plt.hist(test_errors[y_test_full == 1], bins=80, alpha=0.6, label="Fraud",  density=True, color="crimson")
plt.axvline(THRESHOLD, color="black", ls="--", lw=2, label=f"Threshold = {THRESHOLD:.4f}")
plt.yscale("log")
plt.xlabel("Reconstruction error (MSE)")
plt.ylabel("Density (log scale)")
plt.title("Reconstruction error — Normal vs Fraud (test set)")
plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
# 9b. Confusion matrix heatmap (test set).
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues", cbar=False,
    xticklabels=["Normal", "Fraud"],
    yticklabels=["Normal", "Fraud"],
)
plt.title("Confusion Matrix — Test set")
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.tight_layout(); plt.show()

In [ ]:
# 9c. ROC curve and Precision-Recall curve, side by side.
fpr, tpr, _ = roc_curve(y_test_full, test_errors)
prec_curve, rec_curve, _ = precision_recall_curve(y_test_full, test_errors)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(fpr, tpr, lw=2, label=f"AUC = {roc_auc:.4f}")
axes[0].plot([0, 1], [0, 1], "k--", lw=1, alpha=0.6)
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC curve — Test set")
axes[0].legend(loc="lower right")

axes[1].plot(rec_curve, prec_curve, lw=2, label=f"AP = {pr_auc:.4f}")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision–Recall curve — Test set")
axes[1].legend(loc="lower left")

plt.tight_layout(); plt.show()

## 10. Persist artifacts for deployment

We save:

- The trained autoencoder in both **`.keras`** (recommended for TF 2.x) and **`.h5`** (legacy
  format the project rubric asks for).
- The chosen threshold and a few useful metadata fields in a small JSON file so the Flask
  app can load it without any TF dependency.

In [ ]:
autoencoder.save(MODEL_KERAS_PATH)
autoencoder.save(MODEL_H5_PATH)


def _strip(d):
    """Drop non-JSON-serialisable keys (e.g. numpy arrays)."""
    return {k: v for k, v in d.items() if not k.startswith("_") and k != "y_pred"}


threshold_payload = {
    "threshold": float(THRESHOLD),
    "feature_order": FEATURES,
    "n_features": N_FEATURES,
    "selection_strategy": "F2-optimal on validation precision-recall curve (recall weighted 4x precision)",
    "validation_candidates": {
        "f1_optimal":           val_candidates["f1_optimal"],
        "f2_optimal":           val_candidates["f2_optimal"],
        "recall_at_least_0.90": val_candidates["recall_at_least_0.90"],
    },
    "test_metrics": {
        "production_strategy": "f2_optimal",
        "ranking": {"roc_auc": float(roc_auc), "pr_auc": float(pr_auc)},
        "per_strategy": {k: _strip(v) for k, v in results.items()},
    },
}
with open(THRESHOLD_JSON_PATH, "w") as f:
    json.dump(threshold_payload, f, indent=2)

print("Saved artifacts:")
print(" -", MODEL_KERAS_PATH)
print(" -", MODEL_H5_PATH)
print(" -", THRESHOLD_JSON_PATH)
print(" -", HISTORY_JSON_PATH)
print(f"\nProduction threshold = {THRESHOLD:.6f}  "
      f"(test recall = {recall:.4f},  test F1 = {f1:.4f})")

## 11. Deployment-ready inference helper

Self-contained function the Flask web app can call. Given a DataFrame (or 2-D array) of
already-standardised transactions, it returns reconstruction errors and binary predictions
using the saved threshold.

In [ ]:
from tensorflow.keras.models import load_model


def predict_fraud(
    samples,
    model_path: str = str(MODEL_KERAS_PATH),
    threshold_path: str = str(THRESHOLD_JSON_PATH),
):
    """Production-style inference wrapper.

    Parameters
    ----------
    samples : pandas.DataFrame | np.ndarray
        Already-standardised transactions with the exact same feature order used at training.
    model_path : str
        Path to the saved Keras model.
    threshold_path : str
        Path to the threshold JSON written above.

    Returns
    -------
    dict with keys ``errors`` (per-sample MSE), ``predictions`` (0=normal, 1=fraud),
    and ``threshold``.
    """
    with open(threshold_path, "r") as f:
        meta = json.load(f)
    thr = float(meta["threshold"])
    feature_order = meta["feature_order"]

    if isinstance(samples, pd.DataFrame):
        X = samples[feature_order].values.astype(np.float32)
    else:
        X = np.asarray(samples, dtype=np.float32)
        if X.ndim == 1:
            X = X.reshape(1, -1)

    model = load_model(model_path, compile=False)
    X_hat = model.predict(X, verbose=0)
    errors = np.mean(np.square(X - X_hat), axis=1)
    preds = (errors > thr).astype(np.int32)
    return {"errors": errors, "predictions": preds, "threshold": thr}


# Smoke test on a few rows from the test set.
sample_df = pd.DataFrame(X_test_full[:5], columns=FEATURES)
demo = predict_fraud(sample_df)
print("Reconstruction errors :", np.round(demo["errors"], 6))
print("Predictions           :", demo["predictions"])
print("True labels           :", y_test_full[:5])
print("Threshold             :", demo["threshold"])

## 12. Summary & next steps

- The autoencoder was trained **only on normal transactions** — the imbalance is sidestepped.
- We split frauds into a **threshold-selection half** and a **final-test half** so the
  reported numbers are honestly held out.
- Regularisation (L2 + BatchNorm + light Dropout) + EarlyStopping kept the train/val curves
  close while still letting the AE fit the normal manifold tightly.
- We computed **three threshold candidates** on validation:
  - F1-optimal — balanced precision/recall benchmark.
  - **F2-optimal — production choice** (recall weighted 4× precision; right call for fraud).
  - Recall ≥ 0.90 — strictest policy with the highest precision still satisfying that floor.
  All three are evaluated honestly on the test set so reviewers can see the trade-off.
- All deployment artefacts are saved under `./artifacts/`:
  - `autoencoder_fraud.keras` / `autoencoder_fraud.h5` — trained model
  - `threshold.json` — production threshold + all candidates + test metrics
  - `history.json` — training curves
- The `predict_fraud(...)` helper loads the F2-optimal threshold and is ready to be imported
  by the Flask web app.